# 🎬 AI Movie Maker Colab（MiniMax H3-Turbo）— 教學影片／招生影片／OSCE 示範影片

**流程：分鏡腳本（10 秒 3 鏡／15 秒 6 鏡）→ 角色與場景資產 → 每次生成 2 分鐘 → 你確認 → 再生成下一批 → 全片合成**

**腳本格式**（條列文字、不用表格；請 AI 寫腳本的提示詞在 `examples/script_prompt_template.md`）：
```
段落 1（0–10 秒｜參考圖）：自我介紹
場景：病床旁
分鏡 1-1｜全景｜3 秒
- 色調打光：…
- 畫面描述：…
- 旁白對白：（對病人）「…」
- 音效配樂：…；配樂：無
```
一個段落 = 一支 H3 影片（段內分鏡寫成 `[Shot 1] [Shot 2] …` 在同一支裡切鏡）。段落模式：`參考圖`（場景＋角色圖）、`參考圖+首幀`（再用融圖首幀鎖構圖）、`首幀`、`首尾幀`（大幅運鏡、複雜動作）。

| 步驟 | 做什麼 | 需要 GPU？ |
| --- | --- | --- |
| Step 1 | 安裝環境；工作資料夾放在 Google Drive（斷線、隔天都能接著做） | 生成才需要 |
| Step 2 | 上傳場景圖（`<Picture 1>`）、角色圖／三視圖（`<Picture 2..>`）、首幀/尾幀關鍵幀圖 | 否 |
| Step 3 | 上傳腳本（`.txt` `.md` `.docx`、AI Studio 匯出檔）＋角色/場景設定表 → 規格檢查、對齊報告、分段計畫、Prompt 預覽 | 否 |
| Step 4 | 只生成**下一批（預設 ≤ 2 分鐘）**，產出預覽影片、抽幀檢視圖、審核表 | **是** |
| Step 5 | 審核：`approve` 通過，或 `redo_shots` 只重拍指定鏡號（自動換 seed） | 否 |
| Step 6 | 全部批次確認後合成全片（單一聲軌＋繁中字幕） | 否 |

**省 Colab 運算單元的用法**
1. Step 1–3 可以先用 **CPU 執行階段**（不耗 GPU 點數）把腳本、角色對齊、分段計畫和 Prompt 都看過，沒問題再切換 GPU。
2. Step 4 開始前會先檢查每鏡都有參考圖，**有問題在下載模型前就停下**。
3. 一批生成完，審核期間 GPU 仍在計費：要想很久就勾 `AUTO_DISCONNECT_AFTER_BATCH`，或手動「中斷連線並刪除執行階段」。成果都在 Drive，下次重跑 Step 1→3→5 即可接續。
4. 重拍只重做你指定的鏡號；改了腳本某一鏡的文字，也只會重做那一鏡。
5. Step 6 合成全片不需要 GPU。
6. `QUALITY = draft`（參考圖用 `match` 尺寸，較快）先審內容；全部確認後改 `final`（`max` 尺寸）重跑，沿用同樣的 seed。

**連貫性機制**：角色表固定每個人的參考圖、外觀文字、站位與聲音，每鏡依同一順序送入 `<Picture 2..>`；場景表固定每個空間的基準圖；同場景內自動接上一鏡乾淨尾幀（`hybrid`）；跨段落時下一批第一鏡接已確認段落的最後一鏡。

In [ ]:
#@title Step 1. 環境：工作資料夾（Google Drive）、ComfyUI + VideoHelperSuite + AI-Movie
USE_DRIVE_WORKSPACE = True #@param {type:"boolean"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/AI-movie-workspace" #@param {type:"string"}
INSTALL_FASTER_WHISPER = False #@param {type:"boolean"}
REPO_URL = "https://github.com/a017749-chen/AI-movie.git" #@param {type:"string"}
#@markdown 工作資料夾只放腳本、參考圖、生成片段與審核紀錄；模型與 ComfyUI 仍裝在 Colab 本機 `/content`（快、不佔 Drive）。

import shutil
import subprocess
import sys
from pathlib import Path

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    print("PyTorch CUDA:", torch.version.cuda)
    free_gib = round(shutil.disk_usage("/content").free / 2**30, 1)
    print("Free disk GiB (/content):", free_gib)
    assert shutil.disk_usage("/content").free >= 60 * 2**30, f"需要至少 60 GiB 可用磁碟空間，目前僅剩 {free_gib} GiB。"
else:
    print("ℹ️ 目前沒有 GPU：進入「規劃模式」。可以做 Step 2、3、5、6（腳本、對齊、審核、合成），"
          "要生成（Step 4）時再到「執行階段 > 變更執行階段類型」選 GPU（建議 A100）。")

# 1. 繁體中文字型（燒錄字幕）與 ffmpeg
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-noto-cjk"], check=False)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 ffmpeg 與 ffprobe。"

# 2. 只有 GPU 執行階段才安裝 ComfyUI（Comfy-Org/ComfyUI + Kosinkadink/ComfyUI-VideoHelperSuite）
COMFY = Path("/content/ComfyUI")
VHS = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"
SETUP_MARKER = Path("/content/.h3_comfyui_setup_v1")
if HAS_GPU and not SETUP_MARKER.exists():
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"),
         "huggingface_hub", "hf_xet", "pillow", "requests"],
        check=True,
    )
    if not VHS.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(VHS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VHS / "requirements.txt")], check=True)
    SETUP_MARKER.touch()
elif HAS_GPU:
    print("重用本次 Colab session 已安裝的 ComfyUI 環境")

# 3. Clone 或更新 AI-movie (mvkit)
REPO_DIR = Path("/content/AI-movie")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if INSTALL_FASTER_WHISPER:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=False)

# 4. 工作資料夾
if USE_DRIVE_WORKSPACE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE_DIR = Path(DRIVE_WORKSPACE)
else:
    WORKSPACE_DIR = Path("/content/workspace_aimovie")
    print("⚠️ 工作資料夾在 Colab 本機：斷線後生成的片段與審核紀錄都會消失。")
REFS_DIR = WORKSPACE_DIR / "refs"
AUDIO_DIR = WORKSPACE_DIR / "audio"
SCRIPTS_DIR = WORKSPACE_DIR / "scripts"
OUTPUT_DIR = WORKSPACE_DIR / "outputs"
for d in (WORKSPACE_DIR, REFS_DIR, AUDIO_DIR, SCRIPTS_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"[READY] 工作資料夾：{WORKSPACE_DIR}（GPU：{'有' if HAS_GPU else '無，規劃模式'}）")

In [ ]:
#@title Step 2. 上傳參考圖（場景圖 `<Picture 1>`、角色圖 `<Picture 2..N>`）與外部音檔（選填）
UPLOAD_REFERENCE_IMAGES = False #@param {type:"boolean"}
UPLOAD_AUDIO_FILES = False #@param {type:"boolean"}
#@markdown 檔名要和 Step 3 角色/場景設定表的「參考圖」欄一致，例如 `scene_bedside.png`、`char_examinee.png`。
#@markdown 場景圖用實際技能教室照片最穩；角色圖用正面半身、素色背景。真人照片請先取得當事人同意。

from pathlib import Path

if UPLOAD_REFERENCE_IMAGES:
    from google.colab import files
    print("請上傳場景圖與角色圖（可一次多張）：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = REFS_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存參考圖：{dst}")

if UPLOAD_AUDIO_FILES:
    from google.colab import files
    print("請上傳單一主音軌（master_audio 模式）或各鏡旁白 WAV（clip_voiceover 模式）：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = AUDIO_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存音檔：{dst}")

existing_imgs = sorted(p for p in REFS_DIR.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))
if existing_imgs:
    print(f"[OK] 目前可用參考圖（{len(existing_imgs)} 張）：{[p.name for p in existing_imgs]}")
else:
    print("[提醒] 還沒有參考圖。H3 每鏡至少需要一張圖；Step 3 的對齊報告會列出缺哪些。")
default_anchor = existing_imgs[0] if existing_imgs else None

In [ ]:
#@title Step 3. 上傳腳本 + 角色／場景設定 → 對齊報告、2 分鐘分段計畫、Prompt 預覽
PROJECT_TITLE = "OSCE_Demo" #@param {type:"string"}
USE_CASE_PRESET = "osce_demo" #@param ["osce_demo", "teaching", "recruitment", "none"]
SCRIPT_SOURCE = "file_path" #@param ["upload_file", "reuse_last", "file_path", "inline_text"]
SCRIPT_FILE_PATH = "/content/AI-movie/examples/osce_demo/segments.txt" #@param {type:"string"}
BIBLE_SOURCE = "file_path" #@param ["inline_text", "upload_file", "reuse_last", "file_path", "none"]
BIBLE_FILE_PATH = "/content/AI-movie/examples/osce_demo/bible.md" #@param {type:"string"}
BATCH_MINUTES = 2.0 #@param {type:"number"}
SEGMENT_RULE = "use_preset" #@param ["use_preset", "10s_3", "15s_6", "free"]
QUALITY = "draft" #@param ["draft", "final"]
#@markdown `draft`：參考圖用 `match` 尺寸，生成較快，用來審內容；`final`：`max` 尺寸（官方說明可能慢好幾倍），沿用 draft 確認過的 seed 重新生成。

#@markdown ---
#@markdown ### 🔊 聲音與 🛡️ 連貫設定
AUDIO_MODE = "native_h3_audio" #@param ["native_h3_audio", "clip_voiceover", "master_audio"]
MASTER_AUDIO_PATH = "" #@param {type:"string"}
CONTINUITY_MODE = "use_preset" #@param ["use_preset", "hybrid", "chain_last_frame", "anchor"]
INFERENCE_MODE = "reference" #@param ["reference", "first_frame"]
ASPECT_RATIO = "16:9" #@param ["16:9", "9:16", "1:1"]
TAIL_GUARD_SEC = 0.35 #@param {type:"number"}
BASE_SEED = 768914106098754 #@param {type:"integer"}

# inline_text 時使用：段落格式（見最上方說明）；舊的「分鏡 1：場景/畫面/對話」寫法與 Markdown 分鏡表格也能讀
SCRIPT_TEXT = """
段落 1（0–10 秒｜參考圖）：開場
場景：病床旁
分鏡 1-1｜全景｜3 秒
- 色調打光：冷白日光燈，均勻柔和
- 畫面描述：考生從畫面左側走到病床旁，病人坐在床邊
- 旁白對白：無
- 音效配樂：技能教室環境音；配樂：無
分鏡 1-2｜中景｜4 秒
- 色調打光：同上
- 畫面描述：考生面向病人微笑自我介紹
- 旁白對白：（對病人）「您好，我是今天的醫師。」
- 音效配樂：環境音
分鏡 1-3｜近景｜3 秒
- 色調打光：同上
- 畫面描述：考生核對病人手圈
- 旁白對白：（對病人）「請告訴我您的全名。」
- 音效配樂：環境音
"""
# inline_text 時使用：角色表與場景表（格式見 examples/osce_demo/bible.md）
BIBLE_TEXT = """
## 角色
| 代號 | 名稱 | 等級 | 別名 | 外觀 | 參考圖 | 其他參考圖 | 位置 | 聲音 | 主講 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| examinee | examinee doctor | 主要 | 考生, 醫師 | Taiwanese man in his mid twenties, short black hair, white short coat | char_examinee.png | | left of the bed | a calm young male voice in Taiwan Mandarin | 是 |
| patient | standardized patient | 次要 | 病人 | Taiwanese man in his sixties, grey hair, light blue patient gown | char_patient.png | | on the bed, right of frame | a gentle older male voice | |

## 場景
| 代號 | 名稱 | 別名 | 描述 | 參考圖 | 角色 |
| --- | --- | --- | --- | --- | --- |
| bedside | OSCE skills room bedside | 病床旁, 病床, 技能教室 | Bright OSCE skills room, hospital bed on the right | scene_bedside.png | examinee, patient |
"""

import shutil
from mvkit.script_loader import h3_frames, preview_shot, save_project_json
from mvkit.script_import import import_script, import_script_text, SCRIPT_EXTS
from mvkit.alignment import align_project, apply_bible, format_alignment_report, load_bible
from mvkit.presets import PRESETS, apply_preset
from mvkit.batches import apply_review_seeds, batch_overview, next_batch, plan_batches, preflight_batch
from mvkit.segment_script import check_segments

LAST_SCRIPT, LAST_BIBLE = WORKSPACE_DIR / "last_script.txt", WORKSPACE_DIR / "last_bible.txt"


def _upload_one(kind):
    from google.colab import files
    print(f"請上傳{kind}：")
    up = files.upload()
    name = next(iter(up))
    dst = SCRIPTS_DIR / Path(name).name
    dst.write_bytes(up[name])
    return dst


# --- 腳本
if SCRIPT_SOURCE == "upload_file":
    script_path = _upload_one("腳本（.txt / .md / .docx / .json / AI Studio 匯出檔）")
elif SCRIPT_SOURCE == "reuse_last":
    assert LAST_SCRIPT.is_file(), "還沒有上次的腳本，請改用 upload_file"
    script_path = Path(LAST_SCRIPT.read_text(encoding="utf-8").strip())
elif SCRIPT_SOURCE == "inline_text":
    script_path = SCRIPTS_DIR / f"{PROJECT_TITLE}_script.txt"
    script_path.write_text(SCRIPT_TEXT, encoding="utf-8")
else:
    script_path = Path(SCRIPT_FILE_PATH)
    if script_path.is_file() and SCRIPTS_DIR not in script_path.parents:
        shutil.copy2(script_path, SCRIPTS_DIR / script_path.name)
        script_path = SCRIPTS_DIR / script_path.name
LAST_SCRIPT.write_text(str(script_path), encoding="utf-8")

project = import_script(script_path, title=PROJECT_TITLE, audio_mode=AUDIO_MODE,
                        default_reference_image=str(default_anchor or ""))

# --- 角色／場景設定表
bible_path = None
if BIBLE_SOURCE == "upload_file":
    bible_path = _upload_one("角色／場景設定表（.md / .json）")
elif BIBLE_SOURCE == "reuse_last" and LAST_BIBLE.is_file():
    bible_path = Path(LAST_BIBLE.read_text(encoding="utf-8").strip())
elif BIBLE_SOURCE == "inline_text":
    bible_path = SCRIPTS_DIR / f"{PROJECT_TITLE}_bible.md"
    bible_path.write_text(BIBLE_TEXT, encoding="utf-8")
elif BIBLE_SOURCE == "file_path":
    bible_path = Path(BIBLE_FILE_PATH)
if bible_path is not None:
    LAST_BIBLE.write_text(str(bible_path), encoding="utf-8")
    apply_bible(project, load_bible(bible_path))

# --- 用途預設、聲音、連貫設定
apply_preset(project, USE_CASE_PRESET)
if CONTINUITY_MODE != "use_preset":
    project.continuity_mode = CONTINUITY_MODE
project.audio_mode = AUDIO_MODE
project.set_aspect(ASPECT_RATIO)
project.inference_mode = INFERENCE_MODE
project.ref_image_size = "max" if QUALITY == "final" else "match"
project.tail_guard_sec = float(TAIL_GUARD_SEC)
project.base_seed = int(BASE_SEED)
if MASTER_AUDIO_PATH.strip():
    project.master_audio_path = MASTER_AUDIO_PATH.strip()

BASE_DIRS = [REFS_DIR, script_path.parent, Path(bible_path).parent if bible_path else None]
BASE_DIRS = [d for d in BASE_DIRS if d]
report = align_project(project, BASE_DIRS)
project.validate()

WORK_DIR = OUTPUT_DIR / project.title / QUALITY
if QUALITY == "final":
    apply_review_seeds(project, OUTPUT_DIR / project.title / "draft")   # 沿用 draft 確認過的 seed
apply_review_seeds(project, WORK_DIR)          # 之前重拍換過的 seed
rule = PRESETS.get(project.preset, {}).get("segment_rule", "free") if SEGMENT_RULE == "use_preset" else SEGMENT_RULE
save_project_json(project, WORK_DIR / "project.json")
BATCHES = plan_batches(project, float(BATCH_MINUTES) * 60)

total = sum(s.duration_sec for s in project.shots)
print(f"✅ {project.title}：{len(project.shots)} 鏡，約 {total / 60:.1f} 分鐘，{project.width}×{project.height} @ 24fps")
print(f"   用途：{PRESETS.get(project.preset, {}).get('label', '自訂')} | 聲音：{project.audio_mode} | "
      f"連貫：{project.continuity_mode} | 預設模式：{project.inference_mode} | 品質：{QUALITY}")
spec_issues = check_segments(project, rule)
if any(s.beats for s in project.shots):
    print(f"\n=== 腳本規格檢查（{rule}）===")
    print("\n".join("   - " + x for x in spec_issues) if spec_issues else "   ✅ 段落秒數、分鏡數、4 大模塊與景別都符合")
print("\n=== 角色／場景對齊報告 ===")
print(format_alignment_report(project, report))
print(f"\n=== 分段計畫（每段 ≤ {BATCH_MINUTES:g} 分鐘）===")
print(batch_overview(project, WORK_DIR, BATCHES))

upcoming = next_batch(project, WORK_DIR, BATCHES)
if upcoming:
    problems = preflight_batch(project, upcoming, BASE_DIRS)
    if problems:
        print(f"\n❌ {upcoming.label()} 還不能生成（Step 4 會在下載模型前停下）：")
        print("\n".join("   - " + p for p in problems))
    first = next(s for s in project.shots if s.shot_id == upcoming.shot_ids[0])
    print(f"\n--- {upcoming.label()} 第一鏡（{first.shot_id}）送入 H3 的 Prompt ---")
    try:
        if first.gen_mode in ("first_last", "reference_keyframes", "first_frame"):
            print(f"（{first.gen_mode} 模式：實際 Prompt 會依首幀/尾幀在 Step 4 產生，並存在 prompts/ 資料夾）")
        print(preview_shot(project, first, base_dirs=BASE_DIRS))
    except ValueError as e:
        print("Prompt 預覽失敗：", e)
else:
    print("\n🎉 所有批次都已確認，直接執行 Step 6 合成全片。")

In [ ]:
#@title Step 4. 生成下一批（≤ 2 分鐘）→ 預覽影片 + 抽幀檢視圖 + 審核表
LOW_VRAM_MODE = "auto" #@param ["auto", "on", "off"]
AUTO_DISCONNECT_AFTER_BATCH = False #@param {type:"boolean"}
COMPUTE_UNITS_PER_HOUR = 0.0 #@param {type:"number"}
#@markdown `COMPUTE_UNITS_PER_HOUR`：填 Colab「資源」面板顯示的每小時運算單元，就會換算預估花費（0 = 只顯示分鐘）。
#@markdown `AUTO_DISCONNECT_AFTER_BATCH`：生成完自動中斷執行階段（工作資料夾必須在 Drive），審核時不再耗點數。

from IPython.display import Image as IPyImage, Markdown, Video, display
from mvkit.batches import (assemble_batch_preview, batch_status, check_gate, engines_for,
                           estimate_gpu_minutes, next_batch, preflight_batch)

batch = next_batch(project, WORK_DIR, BATCHES)
assert batch is not None, "所有批次都已確認，請執行 Step 6。"
check_gate(project, WORK_DIR, BATCHES, batch)
problems = preflight_batch(project, batch, BASE_DIRS)
assert not problems, "生成前檢查未通過（尚未使用 GPU）：\n" + "\n".join(problems)
print(f"▶ {batch.label()}：{', '.join(batch.shot_ids)}")

if batch_status(project, WORK_DIR, batch) != "generated":
    assert HAS_GPU, "生成需要 GPU：請切換執行階段為 GPU，重跑 Step 1、3 後再執行本步驟。"
    est = estimate_gpu_minutes(project, WORK_DIR, batch.shot_ids)
    if est:
        cost = f"，約 {est['gpu_minutes'] / 60 * COMPUTE_UNITS_PER_HOUR:.1f} 運算單元" if COMPUTE_UNITS_PER_HOUR else ""
        print(f"   預估 GPU 時間：{est['gpu_minutes']} 分鐘{cost}（依目前實測速度）")

    from mvkit.h3_colab_runner import H3ColabRunner
    if "runner" not in globals():
        runner = H3ColabRunner(comfy_dir=COMFY, base_url="http://127.0.0.1:8188")
    # 參考圖模式用 Ref2VA、首幀/首尾幀模式用 FL2VA：只下載這一批需要的模型
    new_engines = [e for e in engines_for(project, batch) if e not in runner.model_sets]
    for e in new_engines:
        models = runner.detect_models(e)
        print(f"選用模型組合：mode={models.mode}, variant={models.variant}, diffusion={models.diffusion}, steps={models.steps}")
    if new_engines:
        runner.ensure_models()
    runner.start_server(low_vram=LOW_VRAM_MODE)
    runner.run_storyboard(project, WORK_DIR, base_dirs=BASE_DIRS, resume=True, shot_ids=set(batch.shot_ids))
    # 先請 ComfyUI 釋放 VRAM，後製與字幕處理更穩定
    runner.free_vram()
else:
    print("   這一批已生成、等待審核：直接重建預覽（不用 GPU）")

preview = assemble_batch_preview(project, batch, WORK_DIR, base_dir=AUDIO_DIR)
print("============================================================")
print(f"👀 請審核 {batch.label()}")
print(f"   預覽影片   : {preview.get('subtitled_mp4') or preview['clean_mp4']}")
print(f"   抽幀檢視圖 : {preview['review_sheet']}")
print(f"   審核表     : {preview['checklist']}")
rest = estimate_gpu_minutes(project, WORK_DIR)
if rest and rest["shots"]:
    cost = f"，約 {rest['gpu_minutes'] / 60 * COMPUTE_UNITS_PER_HOUR:.1f} 運算單元" if COMPUTE_UNITS_PER_HOUR else ""
    print(f"   剩餘 {rest['shots']} 鏡預估 GPU 時間：{rest['gpu_minutes']} 分鐘{cost}")
print("   ➜ 看完到 Step 5：approve（通過）或 redo_shots（填要重拍的鏡號）")
print("============================================================")
if preview.get("review_sheet") and Path(preview["review_sheet"]).is_file():
    display(IPyImage(filename=preview["review_sheet"]))
display(Markdown(Path(preview["checklist"]).read_text(encoding="utf-8")))
display(Video(str(preview.get("subtitled_mp4") or preview["clean_mp4"]), embed=True))

if AUTO_DISCONNECT_AFTER_BATCH:
    if USE_DRIVE_WORKSPACE:
        print("已存到 Drive，中斷執行階段以停止計費。審核後重新連線，執行 Step 1 → 3 → 5。")
        from google.colab import runtime
        runtime.unassign()
    else:
        print("⚠️ 工作資料夾不在 Drive，為避免成果遺失，不自動中斷。")

In [ ]:
#@title Step 5. 審核：通過，或只重拍指定鏡號
REVIEW_DECISION = "show_status" #@param ["show_status", "approve", "redo_shots"]
REDO_SHOT_IDS = "" #@param {type:"string"}
REDO_CHAINED_SHOTS_TOO = False #@param {type:"boolean"}
REVIEW_NOTE = "" #@param {type:"string"}
#@markdown `redo_shots`：填鏡號（逗號分隔，例如 `03, 04`），會換新 seed 只重拍這幾鏡；舊片段移到 `raw_clips/_rejected/` 方便比較。
#@markdown 若想修改畫面描述再重拍，直接改腳本、重跑 Step 3，被改到的鏡頭會自動重拍。

from mvkit.batches import approve_batch, batch_overview, batch_status, next_batch, request_redo

batch = next_batch(project, WORK_DIR, BATCHES)
if REVIEW_DECISION == "approve":
    assert batch is not None, "所有批次都已確認。"
    approve_batch(project, WORK_DIR, batch, note=REVIEW_NOTE)
    print(f"✅ 已確認 {batch.label()}")
    nb = next_batch(project, WORK_DIR, BATCHES)
    print(f"➜ 下一步：執行 Step 4 生成 {nb.label()}" if nb else "➜ 全部確認完成，執行 Step 6 合成全片")
elif REVIEW_DECISION == "redo_shots":
    ids = [x.strip() for x in REDO_SHOT_IDS.replace("，", ",").split(",") if x.strip()]
    assert ids, "請在 REDO_SHOT_IDS 填要重拍的鏡號"
    chained = request_redo(project, WORK_DIR, ids, note=REVIEW_NOTE)
    if chained and REDO_CHAINED_SHOTS_TOO:
        request_redo(project, WORK_DIR, chained, note="chained from " + ",".join(ids))
        ids += chained
        chained = []
    print(f"🔁 將重拍：{', '.join(ids)}（已換新 seed）")
    if chained:
        print(f"   提醒：{', '.join(chained)} 是從被重拍鏡頭的尾幀接續的，接點可能不順；"
              f"要一起重拍就勾 REDO_CHAINED_SHOTS_TOO 再執行一次。")
    print("➜ 下一步：執行 Step 4（只會生成被標記重拍的鏡頭）")
    save_project_json(project, WORK_DIR / "project.json")

print("\n=== 目前進度 ===")
print(batch_overview(project, WORK_DIR, BATCHES))

In [ ]:
#@title Step 6. 全片合成：單一聲軌、繁體中文字幕、品質檢視表與下載（不需要 GPU）
USE_ASR_ALIGNMENT = False #@param {type:"boolean"}
BURN_SUBTITLES = True #@param {type:"boolean"}
ALLOW_UNAPPROVED = False #@param {type:"boolean"}
AUTO_DOWNLOAD_MP4 = True #@param {type:"boolean"}

from IPython.display import Image as IPyImage, Video, display
from mvkit.batches import all_approved, batch_overview
from mvkit.colab_pipeline import assemble_movie

if not all_approved(project, WORK_DIR, BATCHES):
    print(batch_overview(project, WORK_DIR, BATCHES))
    assert ALLOW_UNAPPROVED, "還有批次沒確認。全部 approve 後再合成，或勾 ALLOW_UNAPPROVED 強制合成。"

if "runner" in globals():
    runner.free_vram()

raw_clips = [WORK_DIR / "raw_clips" / f"{s.index:02d}_{s.shot_id}.mp4" for s in project.shots]
missing = [p.name for p in raw_clips if not p.is_file()]
assert not missing, f"缺少片段：{missing}"

manifest = assemble_movie(
    project,
    raw_clips,
    WORK_DIR,
    base_dir=AUDIO_DIR,
    use_asr_alignment=USE_ASR_ALIGNMENT,
    asr_device="cpu",
    burn_subtitles=BURN_SUBTITLES,
)

print("============================================================")
print("✅ 成品輸出與「單一聲軌」驗證通過！")
print(f"   - 乾淨無字幕版 MP4 : {manifest['clean_mp4']}")
print(f"   - 燒錄繁中字幕 MP4 : {manifest['subtitled_mp4']}")
print(f"   - SRT 字幕檔       : {manifest['srt_path']}")
print(f"   - ASS 字幕檔       : {manifest['ass_path']}")
print(f"   - 連貫性檢視圖表   : {manifest['review_sheet']}")
a_info = manifest["clean_stream_verification"]["audio_stream"]
print(f"   - 音軌驗證結果     : 1 條視訊軌 + 1 條音訊軌 ({a_info['codec_name']}, {a_info['sample_rate']} Hz, {a_info['channels']} ch)")
print("============================================================")

if manifest.get("review_sheet") and Path(manifest["review_sheet"]).is_file():
    display(IPyImage(filename=manifest["review_sheet"]))

final_video = manifest.get("subtitled_mp4") or manifest["clean_mp4"]
display(Video(str(final_video), embed=True))

if AUTO_DOWNLOAD_MP4:
    from google.colab import files
    files.download(str(final_video))
    if manifest.get("srt_path"):
        files.download(str(manifest["srt_path"]))